In [4]:
import nibabel as nib
import numpy as np
import Data.IO as dio
import json, os, glob
import SimpleITK as sitk
from pathlib import Path
import pickle


In [2]:
def read_nii_from_file(fname):
    '''
    Read a nii volume and return a 2-tuple with the voxel data (np array) and the voxel2metric tranform
    '''
    #assert os.path.isfile(fname), f"nii file not found {fname}"
    nv = ni.load(fname)
    return nv.get_fdata(), nv.affine
def save_nii_to_file(fname, vol, vox2met):
    ni.save(ni.Nifti1Image(vol, vox2met), fname)

In [3]:
vol, v2m = dio.read_nii_from_file("/path/to/Data/CHRU/P0005/2_3d_tof_4slab__elargi.nii.gz")

NameError: name 'dio' is not defined

In [ ]:
def met2vox_transfrom(affine, point):
    met2vox = np.linalg.inv(affine)
    x = met2vox[:3,:]@np.vstack((point, np.ones(point.shape[1])))
    return x

def vox2met_transform(affine, point):
    point = (affine@np.vstack((point, np.ones(point.shape[1]))))[:3,:]
    return point
    
point_vox = np.array([[10, 19], 
                      [0,  1], 
                      [0,  1], 
])
point_mm = np.array([[-15.950],
                     [-5.276],
                     [-2.348]])
print(point_mm.shape)
#point_mm = vox2met_transform(v2m, point_vox)
#print("Metric space :")
#print(point_mm)
#print("Voxel space :")
print(met2vox_transfrom(v2m, point_mm))

# Boxes to .nii.gz file

In [9]:
def load_pickle(path: Path, **kwargs):
    if isinstance(path, str):
        path = Path(path)
    if not any([fix == path.suffix for fix in [".pickle", ".pkl"]]):
        path = Path(str(path) + ".pkl")

    with open(path, "rb") as f:
        data = pickle.load(f, **kwargs)
    return data
from tqdm import tqdm
def boxes2nii(patients_dir, save_dir, threshold=0.5, spheres=False):
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    case_ids = [os.path.splitext(os.path.basename(f))[0] for f in glob.glob(os.path.join(patients_dir, "*_boxes.pkl"))]
    for cid in tqdm(case_ids):
        if not os.path.isfile(os.path.join(save_dir, f"{cid.replace('_boxes', '')}.nii.gz")):
            res = load_pickle(os.path.join(patients_dir, f"{cid}.pkl"))
            instance_mask = np.zeros(res["original_size_of_raw_data"])

            boxes = res["pred_boxes"]
            scores = res["pred_scores"]
            labels = res["pred_labels"]

            _mask = scores >= threshold
            boxes = boxes[_mask]
            labels = labels[_mask]
            scores = scores[_mask]

            idx = np.argsort(scores)
            scores = scores[idx]
            boxes = boxes[idx]
            labels = labels[idx]

            ################################################
            if spheres:
                D11, D12, D13, D21, D22, D23, D31, D32, D33 = res["itk_direction"]
                S11, S22, S33 = res["itk_spacing"]

                DxS11= (D11*S11);
                DxS12= (D12*S22);
                DxS13= (D13*S33);
                DxS21= (D21*S11);
                DxS22= (D22*S22);
                DxS23= (D23*S33);
                DxS31= (D31*S11);
                DxS32= (D32*S33);
                DxS33= (D33*S33);

            prediction_meta = {}
            for instance_id, (pbox, pscore) in enumerate(zip(boxes, scores), start=1):
                mask_slicing = [slice(int(pbox[0]) + 1, int(pbox[2])),
                                slice(int(pbox[1]) + 1, int(pbox[3])),
                                ]
                if instance_mask.ndim == 3:
                    mask_slicing.append(slice(int(pbox[4]) + 1, int(pbox[5])))
                instance_mask[tuple(mask_slicing)] = pscore
                if spheres:
                    point1, point2 = list(map(int, pbox))[:3], list(map(int, pbox))[3:]
                    point = [(c1+ c2)/2 for c1, c2 in zip(point1, point2) ]

                    Px = res["itk_origin"][0] + ((DxS11*point[0]) + (DxS12*point[1]) + (DxS13*point[2]))
                    Py = res["itk_origin"][1] + ((DxS21*point[0]) + (DxS22*point[1]) + (DxS33*point[2]))
                    Pz = res["itk_origin"][2] + ((DxS31*point[0]) + (DxS32*point[1]) + (DxS33*point[2]))
                    center = [Px, Py, Pz]

                    point1 = [res["itk_origin"][0] + ((DxS11*point1[0]) + (DxS12*point1[1]) + (DxS13*point1[2])), 
                              res["itk_origin"][1] + ((DxS21*point1[0]) + (DxS22*point1[1]) + (DxS33*point1[2])), 
                              res["itk_origin"][2] + ((DxS31*point1[0]) + (DxS32*point1[1]) + (DxS33*point1[2]))]
                    point2 = [res["itk_origin"][0] + ((DxS11*point2[0]) + (DxS12*point2[1]) + (DxS13*point2[2])), 
                              res["itk_origin"][1] + ((DxS21*point2[0]) + (DxS22*point2[1]) + (DxS33*point2[2])), 
                              res["itk_origin"][2] + ((DxS31*point2[0]) + (DxS32*point2[1]) + (DxS33*point2[2]))]
                    radius = np.linalg.norm(np.array(point1) - np.array(point2)) / 2


                    prediction_meta[str(int(instance_id))] = {
                        "center": center,
                        "radius": radius,
                        "confidence": float(pscore),
                    }
            if spheres:
                with open(os.path.join(save_dir, f"{cid.replace('_boxes', '')}.json"), "w") as f:
                    json.dump(prediction_meta, f, indent=4)

            instance_mask_itk = sitk.GetImageFromArray(instance_mask)
            instance_mask_itk.SetOrigin(res["itk_origin"])
            instance_mask_itk.SetDirection(res["itk_direction"])
            instance_mask_itk.SetSpacing(res["itk_spacing"])
            sitk.WriteImage(instance_mask_itk, os.path.join(save_dir, f"{cid.replace('_boxes', '')}.nii.gz"))
            #print(f"{cid} : {np.unique(instance_mask).shape[0]-1} CC.")


threshold = 0.01
patients_dir = "/path/to/0Work/EXTERNAL_VALIDATION/nnDetection/Fold5/test_predictions"
save_dir = os.path.join(patients_dir,"..", "test_predictions_nii_files")
boxes2nii(patients_dir, save_dir, threshold=threshold, spheres=False)


100% 87/87 [03:27<00:00,  2.38s/it]


In [23]:
# boxes to spheres
import pickle
import numpy as np
input_box = "/path/to/0Work/Baselines/nnDetection/CHUV/Fold5/models/Task500_AneurysmsDetection/RetinaUNetV001_D3V001_3d/fold0/test_predictions/sub013ses20101220_boxes.pkl"

# load pkl


with open(input_box, 'rb') as f:
    predictions = pickle.load(f)
# keys = (['pred_boxes', 'pred_scores', 'pred_labels', 'restore', 'original_size_of_raw_data', 'itk_origin', 'itk_spacing', 'itk_direction'])
#box = (x1, x2; y1,y2; z1, z2)

THRESHOLD = 0.01

keep = predictions["pred_scores"] > THRESHOLD

boxes = predictions["pred_boxes"][keep]
scores = predictions["pred_scores"][keep]

centers =  np.stack([(boxes[:, 2] + boxes[:, 0]) / 2., (boxes[:, 3] + boxes[:, 1]) / 2., (boxes[:, 5] + boxes[:, 4]) / 2.], axis=1)
# widths = dist(min(x1, x2), max(x1, x2)) /2 = np.linalg.norm(boxes[:, 0] - boxes[:, 1]) / 2
# heights = dist(min(y1, y2), max(y1, y2))

#radius = np.sqrt(((boxes[:, 2]-boxes[:, 0])*predictions["itk_spacing"][0])**2 + ((boxes[:, 3] + boxes[:, 1])*predictions["itk_spacing"][0])**2 + ((boxes[:, 4] + boxes[:, 4])*predictions["itk_spacing"][0])**2)
#radius = np.linalg.norm(np.array(point1) - np.array(point2)) / 2

print(centers.shape)
print(radius)


(33, 3)
[298.5812  311.02884 340.57166 359.85303 309.05267 216.5841  288.24207
 280.08487 203.99754 334.19116 160.50069 160.64403 318.80182 281.14484
 246.84435 202.52315 308.61295 382.2753  136.76448 235.9469  288.1697
 205.64793 328.52765 296.3563  327.56644 299.94998 237.955   248.23816
 326.9717  298.81134 352.79095 330.62128 259.80316]


In [22]:
predictions["itk_spacing"]
#predictions["itk_direction"]

(0.46875, 0.46875, 0.7000000476837158)

In [ ]:
def yolobbox2bbox(x,y,w,h):
    x1, y1 = x-w/2, y-h/2
    x2, y2 = x+w/2, y+h/2
    return x1, y1, x2, y2

def bbox2yolobbox():
    

# spheres to Bounding boxes

In [ ]:
from Volume.Edition import drawCube

In [ ]:
label_file = "/path/to/0Work/Experiments/Networks/UNet_DS_MaxPool/Predictions/200_epochs/Labels/P0001.json"
pred_file = "/path/to/0Work/Experiments/Networks/UNet_DS_MaxPool/Predictions/200_epochs/P0001.nii.gz"
out_file = "/path/to/0Work/Experiments/Networks/UNet_DS_MaxPool/Predictions/200_epochs/P0001_bboxes.nii.gz"
# load volume with spheres
vol, v2m = dio.read_nii_from_file (pred_file)

confidence_thr = 0.01

In [ ]:

with open(label_file, "r") as f:
    predictions = json.load(f)
if len(predictions) > 0: 
    pred_spheres = np.empty((0, 5)) # x, y, z, radius, confidence
    for key in predictions:
        pred_spheres = np.vstack((pred_spheres, 
                                  np.hstack(( predictions[key]["center"], 
                                              predictions[key]["radius"], 
                                              predictions[key]["confidence"]))))

    # Remove spheres with low confidence score than confidence_thr
    pred_spheres = pred_spheres[(pred_spheres[:, 4] >= confidence_thr), :]
    
    # Draw boxes in volume
    instance_mask = np.zeros(vol.shape)
    for sphere in pred_spheres:
        center = sphere[:3]
        radius = sphere[3]
        score = sphere[4]
        drawCube(instance_mask, v2m, center, radius, val=score)
    dio.save_nii_to_file(out_file, instance_mask, v2m)